# Перенос обучения

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Перенос обучения».

## Научная цель

Перенос обучения позволяет не обучать модель с нуля. Признаки, выученные сетью на большой коллекции данных, переиспользуются для новой задачи с малым числом размеченных примеров. Это резко снижает требования к разметке — что особенно ценно, когда она дорога и требует эксперта.

В этом блокноте мы воспроизведём ключевую идею переноса на компактных данных: сначала «предобучим» сеть-извлекатель признаков на одной выборке цифр, затем перенесём её на родственную задачу с малым числом примеров и сравним с обучением с нуля. Блокнот исполняется на CPU за секунды.

## Интуиция за методом

Представьте, что вы нанимаете опытного микроскописта из соседней лаборатории, чтобы он помог анализировать ваши снимки нового типа клеток. Ему не нужно учить, что такое клетка, ядро или мембрана — он это уже знает. Ему достаточно нескольких часов работы с вашими конкретными препаратами, чтобы начать надёжно различать нужные вам классы.

Перенос обучения работает по той же логике. Нейронная сеть, обученная на миллионах изображений, уже умеет распознавать края, текстуры, формы и структуры — это её «базовые профессиональные навыки». Для вашей задачи нужно лишь переучить финальный классификатор (небольшую «голову»), подстроив его под ваши классы. Это занимает во много раз меньше данных и времени, чем обучение с нуля.

**Ключевые термины, которые встретятся в блокноте:**

- **Предобучение (pre-training)** — обучение модели на большом наборе данных (исходный домен) с целью получить универсальные признаки.
- **Извлекатель признаков (feature extractor)** — часть сети (обычно свёрточные слои), которая преобразует входные данные в компактный набор информативных характеристик.
- **Классификационная голова** — финальный слой или группа слоёв поверх признаков, принимающая решение о классе.
- **Заморозка слоёв (freezing)** — запрет на изменение весов части сети при обучении (`requires_grad = False`). Заморозка извлекателя позволяет обучать только голову.
- **Дообучение (fine-tuning)** — более глубокая адаптация: часть или все слои «размораживают» и обучают с малым темпом, чтобы подстроить предобученные признаки под новый домен, не «стёрев» полезные знания.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch scikit-learn numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор рукописных цифр `digits` из `scikit-learn` (1797 изображений 8x8). Чтобы наглядно показать перенос, разделим его на два **домена**:

- **Исходный домен** — цифры 0–4, много примеров (имитирует «большую коллекцию», на которой предобучена сеть).
- **Целевой домен** — цифры 5–9, мы искусственно оставляем лишь 150 размеченных примеров (имитирует вашу задачу с дефицитом разметки).

**Зачем такое разделение?** Оно воспроизводит типичную ситуацию в науке: есть большая родственная коллекция (например, стандартная база клеток), и есть несколько сотен ваших собственных снимков новых структур. Мы хотим выяснить, помогает ли предобучение на «чужих» данных.

In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

data = load_digits()
X = (data.images.astype('float32') / 16.0)[:, None, :, :]
y = data.target.astype('int64')

# Исходный домен: цифры 0-4.
src = y < 5
Xs, ys = X[src], y[src]
# Целевой домен: цифры 5-9, метки переводим в диапазон 0-4.
tgt = y >= 5
Xt, yt = X[tgt], y[tgt] - 5

# В целевом домене оставляем мало размеченных примеров.
Xt_lab, Xt_test, yt_lab, yt_test = train_test_split(
    Xt, yt, train_size=150, random_state=0, stratify=yt)

print(f'Исходный домен (предобучение): {len(Xs)} примеров')
print(f'Целевой домен: размечено {len(Xt_lab)}, тест {len(Xt_test)}')

## 4. Применение метода

### Шаг 1. Архитектура: разделение на извлекатель и голову

Разобьём сеть на две части. Это ключевой структурный приём переноса обучения:

- `FeatureExtractor` — свёрточный «хребет». Он учится находить общие признаки: края, пятна, текстуры. Именно его веса мы будем **переносить**.
- `Classifier` — лёгкая полносвязная голова. Её задача — интерпретировать найденные признаки применительно к конкретному набору классов. Именно её мы будем обучать заново.

Далее проведём **три** эксперимента и сравним их результаты:
1. Предобучение извлекателя на исходном домене.
2. Перенос: извлекатель заморожен, обучается только голова (150 примеров).
3. Базовый вариант: всё обучается с нуля на 150 примерах (без предобучения).

In [ ]:
import torch.nn as nn


class FeatureExtractor(nn.Module):
    """Свёрточный извлекатель признаков изображения 8x8."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.net(x)


class Classifier(nn.Module):
    """Полносвязная голова поверх признаков."""

    def __init__(self, n_classes=5):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(32 * 2 * 2, 64), nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.head(x)


def train(extractor, classifier, X, y, epochs, lr,
          freeze_extractor=False):
    """Обучает связку извлекатель + классификатор."""
    params = list(classifier.parameters())
    if not freeze_extractor:
        params += list(extractor.parameters())
    opt = torch.optim.Adam(params, lr=lr)
    crit = nn.CrossEntropyLoss()
    Xt_, yt_ = torch.from_numpy(X), torch.from_numpy(y)
    for _ in range(epochs):
        extractor.train(not freeze_extractor)
        classifier.train()
        opt.zero_grad()
        feats = extractor(Xt_)
        loss = crit(classifier(feats), yt_)
        loss.backward()
        opt.step()


def accuracy(extractor, classifier, X, y):
    extractor.eval(); classifier.eval()
    with torch.no_grad():
        pred = classifier(extractor(torch.from_numpy(X))).argmax(1)
    return (pred.numpy() == y).mean()

In [ ]:
# Шаг 1. Предобучение извлекателя на исходном домене (цифры 0-4).
extractor = FeatureExtractor()
src_head = Classifier()
train(extractor, src_head, Xs, ys, epochs=120, lr=2e-3)
print('Извлекатель предобучен на исходном домене.')

# Шаг 2а. Перенос обучения: извлекатель заморожен, обучаем только голову.
import copy
transfer_extractor = copy.deepcopy(extractor)
transfer_head = Classifier()
train(transfer_extractor, transfer_head, Xt_lab, yt_lab,
      epochs=120, lr=2e-3, freeze_extractor=True)
acc_transfer = accuracy(transfer_extractor, transfer_head,
                        Xt_test, yt_test)

# Шаг 2б. Базовый вариант: обучение с нуля только на малом наборе.
scratch_extractor = FeatureExtractor()
scratch_head = Classifier()
train(scratch_extractor, scratch_head, Xt_lab, yt_lab,
      epochs=120, lr=2e-3)
acc_scratch = accuracy(scratch_extractor, scratch_head,
                       Xt_test, yt_test)

print(f'Точность с переносом обучения: {acc_transfer:.3f}')
print(f'Точность при обучении с нуля:  {acc_scratch:.3f}')

### Шаг 2. Визуализация результата — сравнение подходов

Столбчатая диаграмма ниже наглядно показывает выигрыш переноса обучения. Обратите внимание: оба варианта обучены на **одинаковом числе примеров** (150) целевого домена. Разница в точности целиком объясняется качеством стартовых признаков.

### Шаг 3. Сравнение подходов

Столбчатая диаграмма наглядно показывает выигрыш переноса обучения в условиях дефицита размеченных данных.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 5.0))
labels = ['Обучение с нуля', 'Перенос обучения']
values = [acc_scratch, acc_transfer]
bars = ax.bar(labels, values,
              color=[VIZ['series'][2], VIZ['series'][1]], width=0.55)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02,
            f'{v:.3f}', ha='center', fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Доля верных ответов на тесте')
ax.set_title('Перенос обучения против обучения с нуля\n(150 размеченных примеров целевого домена)')
fig.tight_layout()
plt.show()

### Как читать эту диаграмму

Высота каждого столбца — доля верных ответов (от 0 до 1) на тестовой части целевого домена. Оба варианта видели **одинаковые 150 примеров**. Разница в высоте столбцов — это и есть эффект переноса обучения: предобученные признаки дают «фору».

Если столбцы примерно одинаковы — домены слишком далеки друг от друга (признаки не переносятся). Если перенос заметно выше — имеет смысл в реальной задаче воспользоваться предобученной сетью.

### «Ага»-эксперимент: пространство признаков до и после переноса

Ячейка ниже проецирует признаки тестовых примеров целевого домена на плоскость (метод PCA). Слева — пространство признаков **случайно инициализированной** (необученной) сети. Справа — пространство признаков **предобученного** извлекателя. Видно ли разделение классов уже на уровне признаков, до обучения головы?

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Xt_t = torch.from_numpy(Xt_test)

# Признаки случайно инициализированной сети (без предобучения).
random_extractor = FeatureExtractor()
random_extractor.eval()
with torch.no_grad():
    feats_random = random_extractor(Xt_t).numpy()

# Признаки предобученного извлекателя.
transfer_extractor.eval()
with torch.no_grad():
    feats_transfer = transfer_extractor(Xt_t).numpy()

# Проекция на два главных компонента.
pca = PCA(n_components=2, random_state=0)
coords_random = pca.fit_transform(feats_random)
coords_transfer = pca.fit_transform(feats_transfer)

colors_map = [VIZ["series"][i % len(VIZ["series"])] for i in yt_test]
class_labels = [str(c + 5) for c in range(5)]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))
for ax, coords, title in [
        (axes[0], coords_random, "Признаки без предобучения"),
        (axes[1], coords_transfer, "Признаки после предобучения")]:
    for cls in range(5):
        mask = yt_test == cls
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   color=VIZ["series"][cls % len(VIZ["series"])],
                   s=40, alpha=0.75, label=f"Цифра {cls + 5}")
    ax.set_title(title)
    ax.set_xlabel("Главная компонента 1")
    ax.set_ylabel("Главная компонента 2")
    ax.legend(loc="best", fontsize=9)

fig.suptitle("PCA-проекция признаков тестовых примеров целевого домена",
             fontsize=13)
fig.tight_layout()
plt.show()

### Как читать эти графики

Каждая точка — один тестовый снимок, спроецированный на плоскость. Цвет соответствует истинному классу (цифре).

- **Левый график (без предобучения):** точки перемешаны — случайные признаки не несут информации о классе.
- **Правый график (после предобучения):** точки образуют обособленные группы по цвету. Это означает, что предобученные признаки уже разделяют классы целевого домена — даже без обучения финальной головы. Линейному классификатору поверх таких признаков потребуется очень мало примеров, чтобы провести границу между группами.

Этот график — наглядное объяснение, **почему** перенос обучения работает.

## 5. Интерпретация результата

- При малом числе размеченных примеров **перенос обучения обычно заметно точнее** обучения с нуля: извлекатель уже умеет выделять общие признаки, и сети остаётся настроить лишь финальную голову.
- **PCA-карта признаков** показывает, почему это работает: предобученные признаки уже группируют похожие объекты вместе ещё до обучения головы.
- Чем **ближе исходный домен к целевому** и чем **больше своих данных**, тем глубже можно дообучать. При совсем малом наборе извлекатель замораживают; при большем — размораживают часть слоёв и дообучают с малым темпом обучения (fine-tuning).
- Если исходный и целевой домены сильно различаются, выигрыш от переноса падает; в крайнем случае возможен отрицательный перенос.

Для реальных задач компьютерного зрения берут готовые предобученные сети (ResNet, EfficientNet из `torchvision`) — здесь предобучение сделано вручную лишь чтобы показать механизм на быстрых данных.

## 6. Попробуйте на своих данных

Перенос обучения наиболее ценен, когда размеченных данных по вашей задаче мало.

1. Подготовьте небольшой размеченный набор `X` формы `(наблюдение, канал, высота, ширина)` и метки `y`.
2. В реальной работе вместо ручного предобучения возьмите готовую сеть из `torchvision.models` с весами `weights='DEFAULT'`.
3. Замените финальный слой на свой (под нужное число классов) и обучайте его, заморозив остальные через `requires_grad = False`.
4. При достаточном объёме данных разморозьте часть слоёв и дообучите с малым темпом обучения.

In [ ]:
# --- Шаблон переноса с готовой предобученной сети ---
# from torchvision import models
# import torch.nn as nn
#
# net = models.resnet18(weights='DEFAULT')
# for p in net.parameters():           # замораживаем извлекатель
#     p.requires_grad = False
# net.fc = nn.Linear(net.fc.in_features, ВАШЕ_ЧИСЛО_КЛАССОВ)
#
# # Обучается только net.fc; изображения приводят к 3 каналам
# # и нормируют так же, как при предобучении ImageNet.


## 7. Поэкспериментируйте сами

**Эксперимент 1. Объём целевой выборки.**
В ячейке данных измените `train_size=150` на `train_size=30` или `train_size=300`. Как меняется разрыв между переносом и обучением с нуля? При каком объёме он исчезает?

**Эксперимент 2. Дообучение (fine-tuning).**
Добавьте третий вариант: разморозьте извлекатель (удалите `freeze_extractor=True`) и обучайте его совместно с головой на 150 примерах с уменьшенным темпом `lr=5e-4`. Сравните три точности на диаграмме.

**Эксперимент 3. Несвязанные домены.**
Измените разбиение: предобучайте на чётных цифрах (0, 2, 4, 6, 8), тестируйте на нечётных (1, 3, 5, 7, 9). Насколько изменился выигрыш переноса? Что происходит на PCA-карте признаков?

**Эксперимент 4. PCA против случайной инициализации.**
В ячейке с PCA-картой создайте третий график: признаки `transfer_extractor` **до** предобучения (то есть скопируйте инициализированный, но ещё не обученный извлекатель). Покажите все три рядом.

## Краткие выводы

- Перенос обучения позволяет получить высокое качество с **малым числом размеченных примеров** за счёт повторного использования признаков, выученных на больших данных.
- PCA-карта признаков наглядно объясняет механизм: предобученная сеть уже разделяет объекты в пространстве признаков, финальному классификатору остаётся провести границу.
- Замораживайте извлекатель при малом объёме данных; размораживайте (fine-tuning) при большем — но с малым темпом обучения, чтобы не «стереть» полезные знания.
- В практических задачах берите готовые предобученные сети (`torchvision.models`, `huggingface`): они обучены на несравнимо большем разнообразии данных.
- Если исходный и целевой домены мало связаны, перенос может не помочь — оцените PCA-карту признаков, прежде чем выбирать стратегию.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На PCA-карте признаков после предобучения точки целевого домена образуют хорошо разделённые группы ещё до обучения классификационной головы. На что именно это указывает и почему линейному классификатору поверх таких признаков потребуется значительно меньше примеров, чем при обучении с нуля?
2. Вы применяете перенос обучения к задаче классификации гистологических снимков, используя предобученную на ImageNet сеть ResNet-18. Точность при замороженном извлекателе составляет 0.54 — примерно как у случайного угадывания (5 классов). Назовите наиболее вероятную причину слабого переноса и предложите стратегию, которая может помочь.
3. У вас есть 500 размеченных микроскопических изображений новых структур. Вы планируете использовать предобученный ResNet и хотите выбрать между двумя стратегиями: (а) заморозить все слои кроме последнего, (б) разморозить все слои и обучать с `lr = 2e-3`. Какую выбрать и почему выбор (б) в данном случае рискован?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Предобученные признаки уже несут информацию, различающую классы целевого домена: сеть извлекает общие структурные характеристики (края, формы, текстуры), инвариантные к конкретной задаче. Линейному классификатору в таком пространстве достаточно провести разделяющую гиперплоскость, не формируя новое представление с нуля; для этого нужно намного меньше примеров, чем для обучения всей нелинейной функции.
2. Наиболее вероятная причина — большой доменный разрыв: ImageNet состоит из фотографий объектов, а гистологические снимки имеют принципиально иную текстуру, цветовую статистику и структуру. Замороженный извлекатель выдаёт признаки, непригодные для нового домена. Рекомендуется попробовать дообучение (fine-tuning): разморозить несколько верхних слоёв сети и обучать с малым темпом (lr ~ 1e-4), сохраняя нижние слои замороженными.
3. Следует выбрать стратегию (а) или промежуточную: заморозить нижние слои, разморозить верхние и обучать с малым темпом (lr ~ 1e-4 — 5e-5). Стратегия (б) с высоким темпом обучения на всей сети при 500 примерах ведёт к «стиранию» предобученных весов (catastrophic forgetting): сеть быстро подгонится под малый набор и потеряет обобщённые признаки, что обычно ухудшает результат.
</details>
